In [1]:
# ==========================================
# IMPORTAÇÕES
# ==========================================
import pandas as pd

In [29]:
# ==========================================
# LEITURA DOS ARQUIVOS
# ==========================================

# Base principal de candidatos
df = pd.read_csv('/content/Candidatos.csv', low_memory=False)

# Base auxiliar de municípios (TSE x IBGE)
df_municipio = pd.read_csv(
    'https://raw.githubusercontent.com/danielIESB/Trabalho-Projeto-Eleicao/331f19fb7362f45847f748289f2079447ccf82ee/municipio_tse_ibge.csv',
    sep=';',
    encoding='latin1'
)

# Base de vagas
df_vaga = pd.read_csv(
    'https://raw.githubusercontent.com/danielIESB/Trabalho-Projeto-Eleicao/refs/heads/main/Vagas.csv',
    low_memory=False
)


In [4]:
# ==========================================
# FILTRO — APENAS CANDIDATURAS DEFERIDAS
# ==========================================

df_deferido = df[
    (df['situacao'] == 'deferido') |
    (df['situacao'] == 'deferido com recurso')
].copy()

df_deferido.reset_index(drop=True, inplace=True)

print(f"Total de registros no arquivo original : {len(df)}")
print(f"Total de registros deferidos           : {len(df_deferido)}")
print(f"Registros removidos (não deferidos)    : {len(df) - len(df_deferido)}")
print(f"\nDistribuição por situação:")
print(df_deferido['situacao'].value_counts())

Total de registros no arquivo original : 486156
Total de registros deferidos           : 446742
Registros removidos (não deferidos)    : 39414

Distribuição por situação:
situacao
deferido                445584
deferido com recurso      1158
Name: count, dtype: int64


In [5]:
# ==========================================
# TABELA: MUNICIPIO
# ==========================================

# 1. Extrair municípios únicos presentes nas candidaturas deferidas
municipios_candidatos = df_deferido[['id_municipio']].drop_duplicates().dropna().copy()
municipios_candidatos['id_municipio'] = municipios_candidatos['id_municipio'].astype(float).astype(int).astype(str)

# 2. Preparar df_municipio para cruzamento
df_municipio['CD_MUNICIPIO_IBGE'] = df_municipio['CD_MUNICIPIO_IBGE'].astype(float).astype(int).astype(str)

# 3. Cruzar para trazer apenas municípios que aparecem nas candidaturas
df_municipio_limpo = pd.merge(
    municipios_candidatos,
    df_municipio[['CD_MUNICIPIO_IBGE', 'CD_MUNICIPIO_TSE', 'NM_MUNICIPIO_TSE', 'SG_UF']],
    left_on='id_municipio',
    right_on='CD_MUNICIPIO_IBGE',
    how='inner'
).rename(columns={
    'id_municipio'     : 'id_municipio',
    'CD_MUNICIPIO_TSE' : 'cd_municipio_tse',
    'NM_MUNICIPIO_TSE' : 'nm_municipio_tse',
    'SG_UF'            : 'sg_uf'
}).drop(columns=['CD_MUNICIPIO_IBGE'])

# 4. Validação
print("--- DIAGNÓSTICO (MUNICIPIO) ---")
print(f"Municípios únicos nas candidaturas : {len(municipios_candidatos)}")
print(f"Municípios encontrados no cruzamento: {len(df_municipio_limpo)}")
print(f"Municípios não encontrados          : {len(municipios_candidatos) - len(df_municipio_limpo)}")
print(f"Nulos em id_municipio               : {df_municipio_limpo['id_municipio'].isnull().sum()}")
print(f"Duplicados em id_municipio          : {df_municipio_limpo['id_municipio'].duplicated().sum()}")

--- DIAGNÓSTICO (MUNICIPIO) ---
Municípios únicos nas candidaturas : 5568
Municípios encontrados no cruzamento: 5568
Municípios não encontrados          : 0
Nulos em id_municipio               : 0
Duplicados em id_municipio          : 0


In [6]:
df_municipio_limpo.to_csv('/content/Municipio_Limpo.csv', index=False)
print("Sucesso! O arquivo 'Municipio_Limpo.csv' foi gerado e está pronto para o banco.")

Sucesso! O arquivo 'Municipio_Limpo.csv' foi gerado e está pronto para o banco.


In [9]:
# ==========================================
# TABELA: ELEICAO
# ==========================================

# 1. Extrair colunas necessárias e remover duplicatas
mapeamento = {
    "id_eleicao"   : "id_eleicao",
    "ano"          : "ano",
    "tipo_eleicao" : "tipo_eleicao",
    "data_eleicao" : "data_eleicao"
    }

df_eleicao = df_deferido[list(mapeamento.keys())].rename(columns=mapeamento).drop_duplicates().copy()
df_eleicao.reset_index(drop=True, inplace=True)

# 2. Análise de duplicatas
duplicados = df_eleicao.groupby('id_eleicao').size()
duplicados_acima_de_1 = duplicados[duplicados > 1].sort_values(ascending=False)

print(f"Total de id_eleicao com duplicidade: {len(duplicados_acima_de_1)}")
duplicados_acima_de_1

Total de id_eleicao com duplicidade: 1


,0
id_eleicao,
32,44


In [8]:
print(df_deferido.columns.tolist())

['ano', 'id_eleicao', 'tipo_eleicao', 'data_eleicao', 'sigla_uf', 'id_municipio', 'id_municipio_tse', 'titulo_eleitoral', 'cpf', 'sequencial', 'numero', 'nome', 'nome_urna', 'numero_partido', 'sigla_partido', 'cargo', 'situacao', 'data_nascimento', 'idade', 'genero', 'instrucao', 'ocupacao', 'estado_civil', 'nacionalidade', 'sigla_uf_nascimento', 'municipio_nascimento', 'email', 'raca']


In [11]:
# Investigar o id_eleicao duplicado
df_eleicao[df_eleicao['id_eleicao'] == 32].head()

,id_eleicao,ano,tipo_eleicao,data_eleicao
5,32,2008,eleicoes suplementares 2008,2009-03-15
6,32,2008,eleicoes suplementares 2008,2010-03-07
7,32,2008,eleicoes suplementares 2008,2009-12-27
8,32,2008,eleicoes suplementares 2008,2010-11-07
9,32,2008,eleicoes suplementares 2008,2010-05-30


In [12]:
# 3. Combinar aproveitando o primeiro valor válido de cada campo
df_eleicao_combinado = df_eleicao.groupby('id_eleicao').first().reset_index()

print(f"Linhas restantes após combinar: {len(df_eleicao_combinado)}")
print(f"Duplicados restantes          : {df_eleicao_combinado['id_eleicao'].duplicated().sum()}")

Linhas restantes após combinar: 229
Duplicados restantes          : 0


In [13]:
# ==========================================
# 1. DIAGNÓSTICO INICIAL
# ==========================================
nulos_antes      = df_eleicao['id_eleicao'].isnull().sum()
duplicados_antes = df_eleicao['id_eleicao'].duplicated().sum()

print("--- DIAGNÓSTICO INICIAL (ELEIÇÃO) ---")
print(f"Valores nulos encontrados     : {nulos_antes}")
print(f"Valores duplicados encontrados: {duplicados_antes}")
print(f"Total de linhas               : {len(df_eleicao)}\n")

# ==========================================
# 2. LIMPEZA DOS DADOS
# ==========================================

# Passo A: Remover nulos na chave primária
df_eleicao_limpo = df_eleicao_combinado.dropna(subset=['id_eleicao']).copy()

# Passo B: Corrigir sufixo ".0"
df_eleicao_limpo['id_eleicao'] = (
    df_eleicao_limpo['id_eleicao']
    .astype(str)
    .str.split('.')
    .str[0]
    .str.strip()
)

# ==========================================
# 3. VALIDAÇÃO FINAL
# ==========================================
nulos_depois      = df_eleicao_limpo['id_eleicao'].isnull().sum()
duplicados_depois = df_eleicao_limpo['id_eleicao'].duplicated().sum()

print("--- APÓS A LIMPEZA ---")
print(f"Valores nulos restantes     : {nulos_depois} (Esperado: 0)")
print(f"Valores duplicados restantes: {duplicados_depois} (Esperado: 0)")
print(f"Total de eleições únicas    : {len(df_eleicao_limpo)}\n")

--- DIAGNÓSTICO INICIAL (ELEIÇÃO) ---
Valores nulos encontrados     : 0
Valores duplicados encontrados: 43
Total de linhas               : 272

--- APÓS A LIMPEZA ---
Valores nulos restantes     : 0 (Esperado: 0)
Valores duplicados restantes: 0 (Esperado: 0)
Total de eleições únicas    : 229



In [14]:
df_eleicao_limpo.to_csv('/content/Eleicao_Limpo.csv', index=False)
print("Sucesso! O arquivo 'Eleicao_Limpo.csv' foi gerado e está pronto para o banco.")

Sucesso! O arquivo 'Eleicao_Limpo.csv' foi gerado e está pronto para o banco.


In [15]:
# ==========================================
# TABELA: PARTIDO
# ==========================================

# 1. Extrair colunas necessárias
mapeamento = {
    "numero_partido" : "numero_partido",
    "sigla_partido"  : "sigla_partido"
}

df_partido = df_deferido[list(mapeamento.keys())].rename(columns=mapeamento).drop_duplicates(subset=['numero_partido'], keep='first').copy()
df_partido.reset_index(drop=True, inplace=True)

# 2. Análise de duplicatas
duplicados = df_partido.groupby('numero_partido').size()
duplicados_acima_de_1 = duplicados[duplicados > 1].sort_values(ascending=False)

print(f"Total de numero_partido com duplicidade: {len(duplicados_acima_de_1)}")
duplicados_acima_de_1

Total de numero_partido com duplicidade: 0


,0
numero_partido,


In [16]:
# ==========================================
# 1. DIAGNÓSTICO INICIAL
# ==========================================
nulos_antes      = df_partido['numero_partido'].isnull().sum()
duplicados_antes = df_partido['numero_partido'].duplicated().sum()

print("--- DIAGNÓSTICO INICIAL (PARTIDO) ---")
print(f"Valores nulos encontrados     : {nulos_antes}")
print(f"Valores duplicados encontrados: {duplicados_antes}")
print(f"Total de linhas               : {len(df_partido)}\n")

# ==========================================
# 2. LIMPEZA DOS DADOS
# ==========================================

# Passo A: Remover nulos na chave primária
df_partido_limpo = df_partido.dropna(subset=['numero_partido']).copy()

# Passo B: Corrigir sufixo ".0"
df_partido_limpo['numero_partido'] = (
    df_partido_limpo['numero_partido']
    .astype(str)
    .str.split('.')
    .str[0]
    .str.strip()
)

# ==========================================
# 3. VALIDAÇÃO FINAL
# ==========================================
nulos_depois      = df_partido_limpo['numero_partido'].isnull().sum()
duplicados_depois = df_partido_limpo['numero_partido'].duplicated().sum()

print("--- APÓS A LIMPEZA ---")
print(f"Valores nulos restantes     : {nulos_depois} (Esperado: 0)")
print(f"Valores duplicados restantes: {duplicados_depois} (Esperado: 0)")
print(f"Total de partidos únicos    : {len(df_partido_limpo)}\n")

--- DIAGNÓSTICO INICIAL (PARTIDO) ---
Valores nulos encontrados     : 0
Valores duplicados encontrados: 0
Total de linhas               : 39

--- APÓS A LIMPEZA ---
Valores nulos restantes     : 0 (Esperado: 0)
Valores duplicados restantes: 0 (Esperado: 0)
Total de partidos únicos    : 39



In [17]:
df_partido_limpo.to_csv('/content/Partido_Limpo.csv', index=False)
print("Sucesso! O arquivo 'Partido_Limpo.csv' foi gerado e está pronto para o banco.")

Sucesso! O arquivo 'Partido_Limpo.csv' foi gerado e está pronto para o banco.


In [18]:
# ==========================================
# TABELA: PESSOA
# ==========================================

# 1. Extrair colunas necessárias
mapeamento = {
    "titulo_eleitoral" : "titulo_eleitoral",
    "cpf"              : "cpf",
    "nome"             : "nome",
    "data_nascimento"  : "data_nascimento",
    "genero"           : "genero",
    "raca"             : "raca"
}

df_pessoa = df_deferido[list(mapeamento.keys())].rename(columns=mapeamento).copy()

# 2. Análise de duplicatas
duplicados = df_pessoa.groupby('titulo_eleitoral').size()
duplicados_acima_de_1 = duplicados[duplicados > 1].sort_values(ascending=False)

print(f"Total de titulo_eleitoral com duplicidade: {len(duplicados_acima_de_1)}")
print("\nExemplo dos que mais se repetem:")
print(duplicados_acima_de_1.head(20))

Total de titulo_eleitoral com duplicidade: 37873

Exemplo dos que mais se repetem:
titulo_eleitoral
1.074584e+11    7
3.980158e+10    6
1.431531e+10    6
2.732799e+10    6
4.581517e+10    5
2.303435e+10    5
1.019831e+11    5
1.686264e+10    5
1.011062e+10    5
6.233762e+09    5
8.997119e+10    5
1.106169e+10    5
3.454540e+10    5
3.601566e+10    5
1.837819e+10    5
8.464374e+10    5
6.202900e+09    5
1.151294e+10    5
1.050952e+10    5
6.719172e+09    5
dtype: int64


In [19]:
# 3. Investigar um exemplo para verificar complementaridade
exemplo = duplicados_acima_de_1.index[0]
df_pessoa[df_pessoa['titulo_eleitoral'] == exemplo]

,titulo_eleitoral,cpf,nome,data_nascimento,genero,raca
20800,1.074584e+11,9.691019e+09,Paulo Roberto Bufalo,1967-05-23,masculino,NaN
75658,1.074584e+11,9.691019e+09,Paulo Roberto Bufalo,1967-05-23,masculino,branca
229067,1.074584e+11,9.691019e+09,Paulo Roberto Bufalo,1967-05-23,masculino,NaN
261226,1.074584e+11,9.691019e+09,Paulo Roberto Bufalo,1967-05-23,masculino,branca
327096,1.074584e+11,9.691019e+09,Paulo Roberto Bufalo,1967-05-23,masculino,branca
355911,1.074584e+11,9.691019e+09,Paulo Roberto Bufalo,1967-05-23,masculino,branca
442748,1.074584e+11,9.691019e+09,Paulo Roberto Búfalo,1964-05-26,masculino,NaN


In [20]:
# 4. Análise de complementaridade
df_dup_completos = df_pessoa[df_pessoa['titulo_eleitoral'].duplicated(keep=False)]

print("--- ANÁLISE DE COMPLEMENTARIDADE ---")
print(f"Total de linhas com titulo repetido: {len(df_dup_completos)}")

valores_por_coluna = df_dup_completos.groupby('titulo_eleitoral').count()
print("\nExemplo de preenchimento nos duplicados:")
print(valores_por_coluna.head(10))

--- ANÁLISE DE COMPLEMENTARIDADE ---
Total de linhas com titulo repetido: 80360

Exemplo de preenchimento nos duplicados:
                  cpf  nome  data_nascimento  genero  raca
titulo_eleitoral                                          
1741708.0           2     2                2       2     1
2051341.0           2     2                2       2     2
4002461.0           3     3                3       3     3
5791279.0           2     2                2       2     0
5991813.0           2     2                2       2     2
7831287.0           2     2                2       2     0
8632402.0           5     5                5       5     5
9492607.0           2     2                2       2     1
9790221.0           2     2                2       2     1
9831600.0           2     2                2       2     1


In [21]:
# 5. Combinar aproveitando o primeiro valor válido de cada campo
df_pessoa_combinado = df_pessoa.groupby('titulo_eleitoral').first().reset_index()

print(f"Linhas restantes após combinar: {len(df_pessoa_combinado)}")
print(f"Duplicados restantes          : {df_pessoa_combinado['titulo_eleitoral'].duplicated().sum()}")

Linhas restantes após combinar: 404255
Duplicados restantes          : 0


In [22]:
# ==========================================
# 1. DIAGNÓSTICO INICIAL
# ==========================================
nulos_antes      = df_pessoa['titulo_eleitoral'].isnull().sum()
duplicados_antes = df_pessoa['titulo_eleitoral'].duplicated().sum()

print("--- DIAGNÓSTICO INICIAL (PESSOA) ---")
print(f"Valores nulos encontrados     : {nulos_antes}")
print(f"Valores duplicados encontrados: {duplicados_antes}")
print(f"Total de linhas               : {len(df_pessoa)}\n")

# ==========================================
# 2. LIMPEZA DOS DADOS
# ==========================================

# Passo A: Remover nulos na chave primária
df_pessoa_limpo = df_pessoa_combinado.dropna(subset=['titulo_eleitoral']).copy()

# Passo B: Corrigir sufixo ".0"
df_pessoa_limpo['titulo_eleitoral'] = (
    df_pessoa_limpo['titulo_eleitoral']
    .astype(str)
    .str.split('.')
    .str[0]
    .str.strip()
)

# ==========================================
# 3. VALIDAÇÃO FINAL
# ==========================================
nulos_depois      = df_pessoa_limpo['titulo_eleitoral'].isnull().sum()
duplicados_depois = df_pessoa_limpo['titulo_eleitoral'].duplicated().sum()

print("--- APÓS A LIMPEZA ---")
print(f"Valores nulos restantes     : {nulos_depois} (Esperado: 0)")
print(f"Valores duplicados restantes: {duplicados_depois} (Esperado: 0)")
print(f"Total de pessoas únicas     : {len(df_pessoa_limpo)}\n")

--- DIAGNÓSTICO INICIAL (PESSOA) ---
Valores nulos encontrados     : 43
Valores duplicados encontrados: 42486
Total de linhas               : 446742

--- APÓS A LIMPEZA ---
Valores nulos restantes     : 0 (Esperado: 0)
Valores duplicados restantes: 0 (Esperado: 0)
Total de pessoas únicas     : 404255



In [23]:
df_pessoa_limpo.to_csv('/content/Pessoas_Limpo.csv', index=False)
print("Sucesso! O arquivo 'Pessoas_Limpo.csv' foi gerado e está pronto para o banco.")

Sucesso! O arquivo 'Pessoas_Limpo.csv' foi gerado e está pronto para o banco.


In [24]:
# ==========================================
# TABELA: CANDIDATURA
# ==========================================

# 1. Extrair colunas necessárias
mapeamento = {
    "sequencial"       : "sequencial_candidato",
    "titulo_eleitoral" : "titulo_eleitoral",
    "id_eleicao"       : "id_eleicao",
    "numero_partido"   : "numero_partido",
    "id_municipio"     : "id_municipio",
    "cargo"            : "cargo",
    "numero"           : "numero_urna",
    "nome_urna"        : "nome_urna",
    "situacao"         : "situacao"
}

df_candidatos = df_deferido[list(mapeamento.keys())].rename(columns=mapeamento).copy()

# 2. Análise de duplicatas na chave composta
duplicados = df_candidatos.groupby(['titulo_eleitoral', 'id_eleicao']).size()
duplicados_acima_de_1 = duplicados[duplicados > 1].sort_values(ascending=False)

print(f"Total de duplicatas na chave composta (titulo_eleitoral, id_eleicao): {len(duplicados_acima_de_1)}")
duplicados_acima_de_1

Total de duplicatas na chave composta (titulo_eleitoral, id_eleicao): 0


,,0
titulo_eleitoral,id_eleicao,


In [25]:
# ==========================================
# 1. DIAGNÓSTICO INICIAL
# ==========================================
nulos_antes      = df_candidatos['titulo_eleitoral'].isnull().sum()
duplicados_antes = df_candidatos.duplicated(subset=['titulo_eleitoral', 'id_eleicao']).sum()

print("--- DIAGNÓSTICO INICIAL (CANDIDATURA) ---")
print(f"Valores nulos em titulo_eleitoral  : {nulos_antes}")
print(f"Valores duplicados (titulo+eleicao): {duplicados_antes}")
print(f"Total de linhas                    : {len(df_candidatos)}\n")

# ==========================================
# 2. LIMPEZA DOS DADOS
# ==========================================

# Passo A: Remover nulos na chave primária
df_candidatos_limpo = df_candidatos.dropna(subset=['titulo_eleitoral', 'id_eleicao']).copy()

# Passo B: Corrigir sufixo ".0" em colunas numéricas
for coluna in ['sequencial_candidato', 'titulo_eleitoral', 'id_eleicao',
               'numero_partido', 'id_municipio', 'numero_urna']:
    df_candidatos_limpo[coluna] = (
        df_candidatos_limpo[coluna]
        .astype(str)
        .str.split('.')
        .str[0]
        .str.strip()
    )

# Passo C: Substituir 'nan' por None
df_candidatos_limpo['id_municipio'] = df_candidatos_limpo['id_municipio'].replace('nan', None)

# ==========================================
# 3. VALIDAÇÃO FINAL
# ==========================================
nulos_depois      = df_candidatos_limpo['titulo_eleitoral'].isnull().sum()
duplicados_depois = df_candidatos_limpo.duplicated(subset=['titulo_eleitoral', 'id_eleicao']).sum()

print("--- APÓS A LIMPEZA ---")
print(f"Valores nulos restantes     : {nulos_depois} (Esperado: 0)")
print(f"Valores duplicados restantes: {duplicados_depois} (Esperado: 0)")
print(f"Total de candidaturas únicas: {len(df_candidatos_limpo)}\n")

--- DIAGNÓSTICO INICIAL (CANDIDATURA) ---
Valores nulos em titulo_eleitoral  : 43
Valores duplicados (titulo+eleicao): 35
Total de linhas                    : 446742

--- APÓS A LIMPEZA ---
Valores nulos restantes     : 0 (Esperado: 0)
Valores duplicados restantes: 0 (Esperado: 0)
Total de candidaturas únicas: 446699



In [26]:
# ==========================================
# 4. VERIFICAR AMARRAÇÃO COM OUTRAS TABELAS
# ==========================================

# Títulos em candidatura que não existem em pessoa
titulos_sem_pessoa = set(df_candidatos_limpo['titulo_eleitoral']) - set(df_pessoa_limpo['titulo_eleitoral'])
print(f"Títulos em candidatura sem correspondência em pessoa : {len(titulos_sem_pessoa)}")

# Eleições em candidatura que não existem em eleicao
eleicoes_sem_ref = set(df_candidatos_limpo['id_eleicao']) - set(df_eleicao_limpo['id_eleicao'])
print(f"Eleições em candidatura sem correspondência em eleicao: {len(eleicoes_sem_ref)}")

# Partidos em candidatura que não existem em partido
partidos_sem_ref = set(df_candidatos_limpo['numero_partido']) - set(df_partido_limpo['numero_partido'])
print(f"Partidos em candidatura sem correspondência em partido: {len(partidos_sem_ref)}")

Títulos em candidatura sem correspondência em pessoa : 0
Eleições em candidatura sem correspondência em eleicao: 0
Partidos em candidatura sem correspondência em partido: 0


In [27]:
df_candidatos_limpo.to_csv('/content/Candidatos_Limpo.csv', index=False)
print("Sucesso! O arquivo 'Candidatos_Limpo.csv' foi gerado e está pronto para o banco.")

Sucesso! O arquivo 'Candidatos_Limpo.csv' foi gerado e está pronto para o banco.


In [36]:
# ==========================================
# TABELA: VAGA
# ==========================================

# 1. Extrair colunas necessárias
mapeamento = {
    "id_eleicao"   : "id_eleicao",
    "sigla_uf"     : "sigla_uf",
    "id_municipio" : "id_municipio",
    "cargo"        : "cargo",
    "vagas"        : "vagas"
}

df_vaga_mapped = df_vaga[list(mapeamento.keys())].rename(columns=mapeamento).copy()

# 2. Análise de duplicatas na chave composta
duplicados = df_vaga_mapped.groupby(['id_eleicao', 'sigla_uf', 'id_municipio', 'cargo']).size()
duplicados_acima_de_1 = duplicados[duplicados > 1].sort_values(ascending=False)

print(f"Total de duplicatas na chave composta (id_eleicao, sigla_uf, id_municipio, cargo): {len(duplicados_acima_de_1)}")
duplicados_acima_de_1.head(20)

Total de duplicatas na chave composta (id_eleicao, sigla_uf, id_municipio, cargo): 0


,,,,0
id_eleicao,sigla_uf,id_municipio,cargo,


In [37]:
# ==========================================
# 1. DIAGNÓSTICO INICIAL
# ==========================================
nulos_antes      = df_vaga_mapped['id_eleicao'].isnull().sum()
duplicados_antes = df_vaga_mapped.duplicated(subset=['id_eleicao', 'sigla_uf', 'id_municipio', 'cargo']).sum()

print("--- DIAGNÓSTICO INICIAL (VAGA) ---")
print(f"Valores nulos em id_eleicao                      : {nulos_antes}")
print(f"Valores duplicados (eleicao+uf+municipio+cargo)  : {duplicados_antes}")
print(f"Total de linhas                                  : {len(df_vaga_mapped)}\n")

# ==========================================
# 2. LIMPEZA DOS DADOS
# ==========================================

# Passo A: Remover nulos na chave
df_vaga_limpo = df_vaga_mapped.dropna(subset=['id_eleicao', 'cargo']).copy()

# Passo B: Corrigir sufixo ".0"
for coluna in ['id_eleicao', 'id_municipio']:
    df_vaga_limpo[coluna] = (
        df_vaga_limpo[coluna]
        .astype(str)
        .str.split('.')
        .str[0]
        .str.strip()
    )

# Passo C: Substituir 'nan' por None em id_municipio
df_vaga_limpo['id_municipio'] = df_vaga_limpo['id_municipio'].replace('nan', None)

# Passo D: Remover eleições sem correspondência na tabela eleicao
eleicoes_sem_ref = set(df_vaga_limpo['id_eleicao']) - set(df_eleicao_limpo['id_eleicao'])
df_vaga_limpo = df_vaga_limpo[~df_vaga_limpo['id_eleicao'].isin(eleicoes_sem_ref)].copy()

print(f"Registros removidos por eleição sem correspondência: {len(eleicoes_sem_ref)} eleições")

# ==========================================
# 3. VERIFICAR AMARRAÇÃO COM OUTRAS TABELAS
# ==========================================
eleicoes_sem_ref_final = set(df_vaga_limpo['id_eleicao']) - set(df_eleicao_limpo['id_eleicao'])
print(f"Eleições sem correspondência após limpeza: {len(eleicoes_sem_ref_final)} (Esperado: 0)")

# ==========================================
# 4. VALIDAÇÃO FINAL
# ==========================================
nulos_depois      = df_vaga_limpo['id_eleicao'].isnull().sum()
duplicados_depois = df_vaga_limpo.duplicated(subset=['id_eleicao', 'sigla_uf', 'id_municipio', 'cargo']).sum()

print("\n--- APÓS A LIMPEZA ---")
print(f"Valores nulos restantes     : {nulos_depois} (Esperado: 0)")
print(f"Valores duplicados restantes: {duplicados_depois} (Esperado: 0)")
print(f"Total de vagas únicas       : {len(df_vaga_limpo)}\n")

--- DIAGNÓSTICO INICIAL (VAGA) ---
Valores nulos em id_eleicao                      : 51243
Valores duplicados (eleicao+uf+municipio+cargo)  : 34431
Total de linhas                                  : 102216

Registros removidos por eleição sem correspondência: 76 eleições
Eleições sem correspondência após limpeza: 0 (Esperado: 0)

--- APÓS A LIMPEZA ---
Valores nulos restantes     : 0 (Esperado: 0)
Valores duplicados restantes: 0 (Esperado: 0)
Total de vagas únicas       : 50812



In [32]:
# ==========================================
# 1. DIAGNÓSTICO INICIAL
# ==========================================
nulos_antes      = df_vaga_mapped['id_eleicao'].isnull().sum()
duplicados_antes = df_vaga_mapped.duplicated(subset=['id_eleicao', 'id_municipio', 'cargo']).sum()

print("--- DIAGNÓSTICO INICIAL (VAGA) ---")
print(f"Valores nulos em id_eleicao              : {nulos_antes}")
print(f"Valores duplicados (eleicao+municipio+cargo): {duplicados_antes}")
print(f"Total de linhas                          : {len(df_vaga_mapped)}\n")

# ==========================================
# 2. LIMPEZA DOS DADOS
# ==========================================

# Passo A: Remover nulos na chave
df_vaga_limpo = df_vaga_mapped.dropna(subset=['id_eleicao', 'cargo']).copy()

# Passo B: Corrigir sufixo ".0"
for coluna in ['id_eleicao', 'id_municipio']:
    df_vaga_limpo[coluna] = (
        df_vaga_limpo[coluna]
        .astype(str)
        .str.split('.')
        .str[0]
        .str.strip()
    )

# Passo C: Substituir 'nan' por None em id_municipio
df_vaga_limpo['id_municipio'] = df_vaga_limpo['id_municipio'].replace('nan', None)

# ==========================================
# 3. VERIFICAR AMARRAÇÃO COM OUTRAS TABELAS
# ==========================================
eleicoes_sem_ref  = set(df_vaga_limpo['id_eleicao']) - set(df_eleicao_limpo['id_eleicao'])
municipios_sem_ref = set(df_vaga_limpo['id_municipio'].dropna()) - set(df_municipio_limpo['id_municipio'])

print(f"Eleições sem correspondência em eleicao  : {len(eleicoes_sem_ref)}")
print(f"Municípios sem correspondência em municipio: {len(municipios_sem_ref)}")

# ==========================================
# 4. VALIDAÇÃO FINAL
# ==========================================
nulos_depois      = df_vaga_limpo['id_eleicao'].isnull().sum()
duplicados_depois = df_vaga_limpo.duplicated(subset=['id_eleicao', 'id_municipio', 'cargo']).sum()

print("\n--- APÓS A LIMPEZA ---")
print(f"Valores nulos restantes     : {nulos_depois} (Esperado: 0)")
print(f"Valores duplicados restantes: {duplicados_depois} (Esperado: 0)")
print(f"Total de vagas únicas       : {len(df_vaga_limpo)}\n")

--- DIAGNÓSTICO INICIAL (VAGA) ---
Valores nulos em id_eleicao              : 51243
Valores duplicados (eleicao+municipio+cargo): 34921
Total de linhas                          : 102216

Eleições sem correspondência em eleicao  : 76
Municípios sem correspondência em municipio: 0

--- APÓS A LIMPEZA ---
Valores nulos restantes     : 0 (Esperado: 0)
Valores duplicados restantes: 387 (Esperado: 0)
Total de vagas únicas       : 50973



In [38]:
df_vaga_limpo.to_csv('/content/Vaga_Limpo.csv', index=False)
print("Sucesso! O arquivo 'Vaga_Limpo.csv' foi gerado e está pronto para o banco.")

Sucesso! O arquivo 'Vaga_Limpo.csv' foi gerado e está pronto para o banco.
